## Imports

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
import wandb

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device available: ", device)
torch.manual_seed(42)

Device available:  cuda


## Data

In [3]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv")
df.groupby('emotion').size()

# Disgust is imbalanced so we do stratified split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['emotion'])

In [4]:
class CustomDataset(Dataset):
    def __init__(self, df, transform=None):
        pixels = np.stack([
            np.array(p.split(), dtype=np.float32).reshape(1, 48, 48) / 255.0
            for p in df['pixels'].values
        ])
        self.pixels = torch.tensor(pixels)
        self.labels = df['emotion'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.pixels[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [20]:
train_dataset = CustomDataset(train_df, transform=None)
val_dataset = CustomDataset(val_df, transform=None)

In [7]:
BATCH_SIZE = 64
LR = 0.001

In [27]:
train_loader = DataLoader(train_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=4*BATCH_SIZE, shuffle=True,
                        num_workers=4, pin_memory=True, persistent_workers=True)

In [10]:
import wandb
!wandb login



wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Helper Functions

In [11]:
def sanity_check(model, train_loader, model_name, LR, device, optimizer, criterion, n_classes=7, overfit_batch_size=4):
    run = wandb.init(
        entity="ldavi22-free-university-of-tbilisi-",
        project="Emotion Recognition",
        name=f"Sanity check {model_name}",
        config={"learning_rate": LR, "architecture": model_name, "dataset": "fer2013"},
    )

    model_copy = copy.deepcopy(model).to(device)
    optimizer_copy = type(optimizer)(model_copy.parameters(), lr=LR)

    images, labels = next(iter(train_loader))
    images, labels = images.to(device), labels.to(device)

    model_copy.eval()
    with torch.no_grad():
        output = model_copy(images)
        loss_init = criterion(output, labels)
        probs = torch.softmax(output, dim=1)
        expected_loss = -torch.log(torch.tensor(1.0 / n_classes)).item()

        wandb.log({
            "init_check/loss": loss_init.item(),
            "init_check/expected_loss": expected_loss,
            "init_check/sample_prob": probs[0][0].item(),
            "init_check/prob_sum": probs[0].sum().item(),
        })
        print(f"Loss @ init: {loss_init.item():.4f} | Expected (-log(1/{n_classes})): {expected_loss:.4f}")

    model_copy.train()
    tiny_images = images[:overfit_batch_size]
    tiny_labels = labels[:overfit_batch_size]

    for step in range(200):
        output = model_copy(tiny_images)
        loss = criterion(output, tiny_labels)
        optimizer_copy.zero_grad()
        loss.backward()
        optimizer_copy.step()
        acc = (output.argmax(dim=1) == tiny_labels).float().mean()
        wandb.log({"overfit/loss": loss.item(), "overfit/acc": acc.item(), "step": step})

    wandb.finish()

In [13]:
def train_model(model, train_loader, val_loader, criterion, optimizer,
                scheduler=None, epochs=50, run_name="run",
                project="Emotion Recognition",
                entity="ldavi22-free-university-of-tbilisi-",
                config_extra=None, checkpoint_path="best_model.pt",
                augment_fn=None, device=None):

    config = {"epochs": epochs}
    if config_extra:
        config.update(config_extra)

    run = wandb.init(entity=entity, project=project, name=run_name, config=config)
    wandb.watch(model, criterion, log="all", log_freq=10)

    n_train_samples = len(train_loader.dataset)
    n_val_samples = len(val_loader.dataset)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        total_loss_train = 0
        total_acc_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if augment_fn is not None:
                inputs = augment_fn(inputs)

            output = model(inputs)
            loss = criterion(output, labels)
            total_loss_train += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            total_acc_train += (torch.argmax(output, axis=1) == labels).sum().item()

        model.eval()
        total_loss_val = 0
        total_acc_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                output = model(inputs)
                loss = criterion(output, labels)
                total_loss_val += loss.item()
                total_acc_val += (torch.argmax(output, axis=1) == labels).sum().item()

        avg_train_loss = total_loss_train / len(train_loader)
        avg_val_loss = total_loss_val / len(val_loader)
        avg_train_acc = (total_acc_train / n_train_samples) * 100
        avg_val_acc = (total_acc_val / n_val_samples) * 100

        history["train_loss"].append(avg_train_loss)
        history["train_acc"].append(avg_train_acc)
        history["val_loss"].append(avg_val_loss)
        history["val_acc"].append(avg_val_acc)

        if scheduler is not None:
            scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)

        wandb.log({
            "train/loss": avg_train_loss,
            "train/acc": avg_train_acc,
            "val/loss": avg_val_loss,
            "val/acc": avg_val_acc,
            "lr": optimizer.param_groups[0]["lr"],
            "epoch": epoch
        })

    # artifact name uses checkpoint filename — no special characters
    artifact = wandb.Artifact(name=checkpoint_path.replace(".pt", ""), type="model")
    artifact.add_file(checkpoint_path)
    wandb.log_artifact(artifact)

    wandb.finish()
    return history

## Model

In [23]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_a = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.bn_a   = nn.BatchNorm2d(out_channels)
        self.conv_b = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.bn_b   = nn.BatchNorm2d(out_channels)
        self.relu   = nn.ReLU()

    
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        x = self.relu(self.bn_a(self.conv_a(x)))
        x = self.bn_b(self.conv_b(x))
        x = self.relu(x + identity)     # add before final relu
        return x


class Net(nn.Module):
    def __init__(self, dropout2d=0.2, dropout_fc=0.5):
        super().__init__()

        # Block 1: 48x48 -> 24x24, 1->64
        self.block1 = ResBlock(1,   64)
        # Block 2: 24x24 -> 12x12, 64->128
        self.block2 = ResBlock(64,  128)
        # Block 3: 12x12 -> 6x6,  128->256
        self.block3 = ResBlock(128, 256)
        # Block 4: 6x6   -> 3x3,  256->512
        self.block4 = ResBlock(256, 512)

        self.pool      = nn.MaxPool2d(2, 2)
        self.dropout2d = nn.Dropout2d(dropout2d)
        self.dropout   = nn.Dropout(dropout_fc)
        self.flatten   = nn.Flatten()

        self.fc     = nn.Linear(512 * 3 * 3, 512)  # 4608 -> 512
        self.output = nn.Linear(512, 7)

    def forward(self, x):
        x = self.block1(x)
        x = self.pool(x)        # 24x24
        x = self.dropout2d(x)

        x = self.block2(x)
        x = self.pool(x)        # 12x12
        x = self.dropout2d(x)

        x = self.block3(x)
        x = self.pool(x)        # 6x6
        x = self.dropout2d(x)

        x = self.block4(x)
        x = self.pool(x)        # 3x3
        x = self.dropout2d(x)

        x = self.flatten(x)     # 4608
        x = torch.relu(self.fc(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

model = Net().to(device)
summary(model, (1, 48, 48))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 48, 48]             128
       BatchNorm2d-2           [-1, 64, 48, 48]             128
            Conv2d-3           [-1, 64, 48, 48]             640
       BatchNorm2d-4           [-1, 64, 48, 48]             128
              ReLU-5           [-1, 64, 48, 48]               0
            Conv2d-6           [-1, 64, 48, 48]          36,928
       BatchNorm2d-7           [-1, 64, 48, 48]             128
              ReLU-8           [-1, 64, 48, 48]               0
          ResBlock-9           [-1, 64, 48, 48]               0
        MaxPool2d-10           [-1, 64, 24, 24]               0
        Dropout2d-11           [-1, 64, 24, 24]               0
           Conv2d-12          [-1, 128, 24, 24]           8,320
      BatchNorm2d-13          [-1, 128, 24, 24]             256
           Conv2d-14          [-1, 128,

## Sanity Check

In [15]:
torch.manual_seed(42)
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

sanity_check(model, train_loader, "ResNet-style", LR, device, optimizer, criterion)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ldavi22 (ldavi22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Loss @ init: 1.9434 | Expected (-log(1/7)): 1.9459


init_check/expected_loss,▁
init_check/loss,▁
init_check/prob_sum,▁
init_check/sample_prob,▁
overfit/acc,▁▁████▁█████████████████████████████████
overfit/loss,█▂▁▂▁▅▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
init_check/expected_loss,1.94591
init_check/loss,1.94339
init_check/prob_sum,1
init_check/sample_prob,0.14072


## Run 1 - Baseline (no scheduler, no augmentation)

In [16]:
torch.manual_seed(42)
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, epochs=80,
                 run_name="ResNet-style baseline",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "ResNet-style (4 blocks, 64-128-256-512)",
            "dataset": "fer2013",
            "epochs": 80,
            "scheduler": "None",
            "augmentation": "None",
            "batch_size": 4*BATCH_SIZE,
        }, checkpoint_path="resnet_r1.pt",
        device=device)

KeyboardInterrupt: 

## Run 2 - + Scheduler

In [18]:
torch.manual_seed(42)
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=7, min_lr=1e-5
)

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=100,
                 run_name="ResNet-style + Scheduler",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "ResNet-style (4 blocks, 64-128-256-512)",
            "dataset": "fer2013",
            "epochs": 100,
            "scheduler": "ReduceLROnPlateau f=0.7 p=7",
            "augmentation": "None",
            "batch_size": 4*BATCH_SIZE,
        }, checkpoint_path="resnet_r2.pt",
        device=device)

epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▂▂▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇███████████████
train/loss,█▇▇▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▂▃▄▅▆▆▆▆▇▇▇▇▇▇▇▇█▇██▇█████████████████
val/loss,▅▅▅▄▃▂▂▂▂▁▁▁▁▁▁▁▂▁▁▁▂▃▂▂▄▄▄▄▄▄▅▅▆▆▆▇▇█▇▇
epoch,57
lr,0.001
train/acc,94.91009
train/loss,0.15928
val/acc,62.92233


KeyboardInterrupt: 

## Run 3 - + Augmentation

In [22]:
torch.manual_seed(42)
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=7, min_lr=1e-5
)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
])

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=120,
                 run_name="ResNet-style + Scheduler + Augmentation",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "ResNet-style (4 blocks, 64-128-256-512)",
            "dataset": "fer2013",
            "epochs": 120,
            "scheduler": "ReduceLROnPlateau f=0.7 p=7",
            "augmentation": "flip+rotation+translate",
            "batch_size": 4*BATCH_SIZE,
        }, checkpoint_path="resnet_r3.pt",
        device=device, augment_fn=train_transform)

KeyboardInterrupt: 

## Run 4 - + Extended Augmentation

In [26]:
torch.manual_seed(42)
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=7, min_lr=1e-5
)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
])

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=120,
                 run_name="ResNet-style + Scheduler + Extended Augmentation",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "ResNet-style (4 blocks, 64-128-256-512)",
            "dataset": "fer2013",
            "epochs": 120,
            "scheduler": "ReduceLROnPlateau f=0.7 p=7",
            "augmentation": "flip+rotation+translate+colorjitter+erasing",
            "batch_size": 4*BATCH_SIZE,
        }, checkpoint_path="resnet_r4.pt",
        device=device, augment_fn=train_transform)

epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇████
lr,██████████████████████████▅▅▅▅▅▅▅▃▃▃▂▂▂▁
train/acc,▁▁▂▃▃▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█████
train/loss,█▇▇▆▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁
val/acc,▁▁▂▃▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████▇▇████████
val/loss,██▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂
epoch,90
lr,0.00024
train/acc,81.02059
train/loss,0.51584
val/acc,68.59979


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b693e929800>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7b693e929800>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

wandb: ERROR Control-C detected -- Run data was not synced


KeyboardInterrupt: 

In [28]:
torch.manual_seed(42)
model = Net(dropout2d=0.3, dropout_fc=0.5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.7, patience=7, min_lr=1e-5
)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
])

result = train_model(model, train_loader, val_loader, criterion, optimizer,
                 scheduler=scheduler, epochs=120,
                 run_name="ResNet-style + Scheduler + Extended Augmentation + Heavy Regularization",
                 project="Emotion Recognition",
                 entity="ldavi22-free-university-of-tbilisi-",
                 config_extra={
            "learning_rate": LR,
            "architecture": "ResNet-style (4 blocks, 64-128-256-512)",
            "dataset": "fer2013",
            "epochs": 120,
            "scheduler": "ReduceLROnPlateau f=0.7 p=7",
            "augmentation": "flip+rotation+translate+colorjitter+erasing",
            "batch_size": 4*BATCH_SIZE,
        }, checkpoint_path="resnet_r4.pt",
        device=device, augment_fn=train_transform)

epoch,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
lr,██████████████████████████████▅▅▅▅▃▂▂▂▁▁
train/acc,▁▁▂▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇█
train/loss,█▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁
val/acc,▁▁▂▄▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████▇█████████
val/loss,█▇▆▅▅▄▄▃▃▃▃▂▂▂▃▂▂▂▂▂▁▂▁▁▁▁▂▁▁▁▁▁▁▂▁▁▁▁▁▁
epoch,119
lr,0.00024
train/acc,75.97422
train/loss,0.65423
val/acc,69.00035
